# Rotated ALMA upsampled multi-colour channel maps with JWST overviews

Select one line below and fill a 5-column x 6-row grid with NIRCam and MIRI in the upper-left slots and 28 factor-two multi-colour velocity panels in the remaining slots.

In [48]:
from pathlib import Path

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
from matplotlib.colors import hsv_to_rgb
from matplotlib.image import imread
from matplotlib.patches import Ellipse
from PIL import Image
from scipy.ndimage import binary_erosion, binary_propagation, map_coordinates

ALMA_DIR = Path('/Users/abarnes/Library/CloudStorage/Dropbox/Data/Galactic/JWST/cloud_h/alma')
PLOT_DIR = Path('/Users/abarnes/Library/CloudStorage/Dropbox/GitHub/cloudh_outflows/plots')
CACHE_DIR = Path('/tmp/cloudh_outflows_channel_maps')
JWST_DIR = Path('/Users/abarnes/Library/CloudStorage/Dropbox/Data/Galactic/JWST/cloud_h/jwst/fits/nanfilled/reprojected')

MIRI_JPEG = JWST_DIR / 'miri_photoshop.jpeg'
MIRI_WCS_FITS = JWST_DIR / 'jw07230-o002_t003_miri_f770w_i2d_nanfilled_reprojected.fits'
NIRCAM_JPEG = JWST_DIR / 'nircam_photoshop.jpeg'
NIRCAM_WCS_FITS = JWST_DIR / 'jw07230-o004_t003_nircam_clear-f356w_i2d_nanfilled_reprojected.fits'
MIRI_WHITE_THRESHOLD = 245

LINE_CUBES = {
    'CO_3-2': {
        'label': r'CO (3-2)',
        'filename': 'member.uid___A001_X3819_X177._035.522-00.274__sci.spw25.cube.I.selfcal.pbcor_CO_3-2_cube_-50_to_150_kms.fits',
    },
    'SiO_8-7': {
        'label': r'SiO (8-7)',
        'filename': 'member.uid___A001_X3819_X177._035.522-00.274__sci.spw27.cube.I.selfcal.pbcor_SiO_8-7_cube_-50_to_150_kms.fits',
    },
    'SO_6_5-5_4': {
        'label': r'SO $6_5-5_4$',
        'filename': 'member.uid___A001_X3621_X2808.Clump_H6_sci.spw31.cube.I.pbcor_SO_6_5-5_4_cube_-50_to_150_kms.fits',
    },
    '13CO_2-1': {
        'label': r'$^{13}$CO (2-1)',
        'filename': 'member.uid___A001_X3621_X2808.Clump_H6_sci.spw33.cube.I.pbcor_13CO_2-1_cube_-50_to_150_kms.fits',
    },
    '12CO_2-1': {
        'label': r'$^{12}$CO (2-1)',
        'filename': 'member.uid___A001_X3621_X2808.Clump_H6_sci.spw43.cube.I.pbcor_12CO_2-1_cube_-50_to_150_kms.fits',
    },
    'SiO_2-1': {
        'label': r'SiO (2-1)',
        'filename': 'member.uid___A001_X879_X395.CloudH_sci.spw27.cube.I.pbcor_SiO_2-1_cube_-50_to_150_kms.fits',
    },
    'CH3OH_2_02-1_01': {
        'label': r'CH$_3$OH $2(0,2)-1(0,1)^{++}$',
        'filename': 'member.uid___A001_X879_X395.CloudH_sci.spw33.cube.I.pbcor_CH3OH_2(0,2)-1(0,1)++_cube_-50_to_150_kms.fits',
    },
    'CS_2-1': {
        'label': r'CS (2-1)',
        'filename': 'member.uid___A001_X879_X395.CloudH_sci.spw37.cube.I.pbcor_CS_2-1_cube_-50_to_150_kms.fits',
    },
}

# SELECTED_LINE = 'CO_3-2'
SELECTED_LINE = 'SiO_8-7'
# SELECTED_LINE = 'SO_6_5-5_4'
# SELECTED_LINE = '13CO_2-1'
# SELECTED_LINE = '12CO_2-1'
# SELECTED_LINE = 'SiO_2-1'
# SELECTED_LINE = 'CH3OH_2_02-1_01'
# SELECTED_LINE = 'CS_2-1'

VELOCITY_MIN = -30.0  # km/s
VELOCITY_MAX = 150.0  # km/s
N_PANELS = 28
UPSAMPLE_FACTOR = 2
STRETCH = 'asinh'  # Choose 'asinh', 'linear', or 'sqrt'.
UPPER_PERCENTILE = 99.7
ASINH_SOFTENING = 0.1
DISPLAY_PA_DEG = 354.80557109226515 - 360

Image.MAX_IMAGE_PIXELS = None

config = LINE_CUBES[SELECTED_LINE]
cube_path = ALMA_DIR / config['filename']
cache_path = CACHE_DIR / f'{SELECTED_LINE}_channel_maps_upsampled.fits'
N_FINE_MAPS = N_PANELS * UPSAMPLE_FACTOR
N_PANEL_COLOURS = 3 * UPSAMPLE_FACTOR
display_velocity_edges = np.linspace(VELOCITY_MIN, VELOCITY_MAX, N_PANELS + 1)
fine_velocity_edges = np.linspace(VELOCITY_MIN, VELOCITY_MAX, N_FINE_MAPS + 1)
fine_velocity_spacing = fine_velocity_edges[1] - fine_velocity_edges[0]

if STRETCH not in {'asinh', 'linear', 'sqrt'}:
    raise ValueError("STRETCH must be 'asinh', 'linear', or 'sqrt'.")
if not cube_path.is_file():
    raise FileNotFoundError(cube_path)

CACHE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'

print(f'Selected {config["label"]}: {cube_path.name}')
print(
    f'{N_PANELS} displayed bins from {VELOCITY_MIN:g} to {VELOCITY_MAX:g} km/s; '
    f'{N_FINE_MAPS} fine maps at {fine_velocity_spacing:g} km/s'
)

Selected SiO (8-7): member.uid___A001_X3819_X177._035.522-00.274__sci.spw27.cube.I.selfcal.pbcor_SiO_8-7_cube_-50_to_150_kms.fits
28 displayed bins from -30 to 150 km/s; 56 fine maps at 3.21429 km/s


In [49]:
def native_velocity_geometry(header, n_channels):
    spectral_wcs = WCS(header).spectral
    centers = spectral_wcs.pixel_to_world(np.arange(n_channels)).to_value(u.km / u.s)
    if n_channels < 2 or not np.all(np.isfinite(centers)):
        raise ValueError('The cube needs at least two finite spectral channels.')

    edges = np.empty(n_channels + 1, dtype=float)
    edges[1:-1] = 0.5 * (centers[:-1] + centers[1:])
    edges[0] = centers[0] - 0.5 * (centers[1] - centers[0])
    edges[-1] = centers[-1] + 0.5 * (centers[-1] - centers[-2])
    return np.minimum(edges[:-1], edges[1:]), np.maximum(edges[:-1], edges[1:])


def cache_is_current(cache_file, source_file, requested_edges, n_fine_maps, upsample_factor):
    if not cache_file.is_file():
        return False

    source_stat = source_file.stat()
    source_header = fits.getheader(source_file)
    expected_shape = (n_fine_maps, source_header['NAXIS2'], source_header['NAXIS1'])
    expected_spacing = requested_edges[1] - requested_edges[0]

    try:
        with fits.open(cache_file, memmap=True) as hdul:
            header = hdul[0].header
            table = hdul['CHANNELS'].data
            checks = [
                hdul[0].data.shape == expected_shape,
                header.get('SRCFILE') == str(source_file),
                header.get('SRCSIZE') == source_stat.st_size,
                np.isclose(header.get('SRCMTIM', np.nan), source_stat.st_mtime, atol=1e-5),
                header.get('CHNBIN') == n_fine_maps,
                header.get('UPSAMP') == upsample_factor,
                np.isclose(header.get('CHVMIN', np.nan), requested_edges[0]),
                np.isclose(header.get('CHVMAX', np.nan), requested_edges[-1]),
                np.isclose(header.get('CHDV', np.nan), expected_spacing),
                len(table) == n_fine_maps,
                np.allclose(table['VMIN'], requested_edges[:-1]),
                np.allclose(table['VMAX'], requested_edges[1:]),
            ]
            return all(checks)
    except (OSError, KeyError, TypeError, ValueError):
        return False


def make_channel_map_cache(source_file, cache_file, requested_edges, upsample_factor):
    with fits.open(source_file, memmap=True) as hdul:
        source_data = hdul[0].data
        source_header = hdul[0].header.copy()
        if source_data.ndim != 3:
            raise ValueError(f'{source_file.name} must contain one 3-D cube, got {source_data.shape}.')
        if not u.Unit(source_header['BUNIT']).is_equivalent(u.K):
            raise ValueError(f'{source_file.name} must be in K, got {source_header["BUNIT"]}.')

        n_channels, ny, nx = source_data.shape
        native_lower, native_upper = native_velocity_geometry(source_header, n_channels)
        channel_maps = np.empty((len(requested_edges) - 1, ny, nx), dtype=np.float32)

        for bin_index, (lower, upper) in enumerate(zip(requested_edges[:-1], requested_edges[1:])):
            overlaps = np.maximum(
                0.0, np.minimum(native_upper, upper) - np.maximum(native_lower, lower)
            )
            channel_indices = np.flatnonzero(overlaps > 0)
            if channel_indices.size == 0:
                channel_maps[bin_index].fill(np.nan)
                print(
                    f'No spectral coverage for {bin_index + 1:02d}/{len(requested_edges) - 1}: '
                    f'{lower:+.1f} to {upper:+.1f} km/s; writing an all-NaN map'
                )
                continue

            integrated = np.zeros((ny, nx), dtype=np.float64)
            valid_any = np.zeros((ny, nx), dtype=bool)
            for channel_index in channel_indices:
                plane = np.asarray(source_data[channel_index], dtype=np.float32)
                finite = np.isfinite(plane)
                integrated[finite] += plane[finite] * overlaps[channel_index]
                valid_any |= finite
            integrated[~valid_any] = np.nan
            channel_maps[bin_index] = integrated.astype(np.float32)
            print(
                f'Integrated {bin_index + 1:02d}/{len(requested_edges) - 1}: '
                f'{lower:+.1f} to {upper:+.1f} km/s '
                f'({channel_indices.size} native channels)'
            )

    source_stat = source_file.stat()
    cache_header = source_header.copy()
    cache_header['BUNIT'] = (u.K * u.km / u.s).to_string(format='fits')
    cache_header['CTYPE3'] = 'VRAD'
    cache_header['CUNIT3'] = 'km/s'
    cache_header['CRPIX3'] = 1.0
    cache_header['CRVAL3'] = 0.5 * (requested_edges[0] + requested_edges[1])
    cache_header['CDELT3'] = requested_edges[1] - requested_edges[0]
    cache_header['CHVMIN'] = (requested_edges[0], 'First channel-map edge in km/s')
    cache_header['CHVMAX'] = (requested_edges[-1], 'Last channel-map edge in km/s')
    cache_header['CHDV'] = (requested_edges[1] - requested_edges[0], 'Channel-map width in km/s')
    cache_header['CHNBIN'] = (len(requested_edges) - 1, 'Number of fine channel maps')
    cache_header['UPSAMP'] = (upsample_factor, 'Fine maps per displayed velocity bin')
    cache_header['SRCFILE'] = (str(source_file), 'Source cube')
    cache_header['SRCSIZE'] = (source_stat.st_size, 'Source cube size in bytes')
    cache_header['SRCMTIM'] = (source_stat.st_mtime, 'Source modification time')

    channels_hdu = fits.BinTableHDU.from_columns(
        [
            fits.Column(name='INDEX', format='J', array=np.arange(len(requested_edges) - 1)),
            fits.Column(name='VMIN', format='D', unit='km/s', array=requested_edges[:-1]),
            fits.Column(name='VMAX', format='D', unit='km/s', array=requested_edges[1:]),
            fits.Column(
                name='VCENTER', format='D', unit='km/s',
                array=0.5 * (requested_edges[:-1] + requested_edges[1:]),
            ),
            fits.Column(
                name='HASDATA', format='L',
                array=np.any(np.isfinite(channel_maps), axis=(1, 2)),
            ),
        ],
        name='CHANNELS',
    )
    temporary_file = cache_file.with_name(f'{cache_file.stem}.writing.fits')
    fits.HDUList([fits.PrimaryHDU(channel_maps, header=cache_header), channels_hdu]).writeto(
        temporary_file, overwrite=True, output_verify='silentfix'
    )
    temporary_file.replace(cache_file)
    print(f'Saved channel-map cache: {cache_file}')


if cache_is_current(
    cache_path, cube_path, fine_velocity_edges, N_FINE_MAPS, UPSAMPLE_FACTOR
):
    print(f'Reusing channel-map cache: {cache_path}')
else:
    make_channel_map_cache(cube_path, cache_path, fine_velocity_edges, UPSAMPLE_FACTOR)

Reusing channel-map cache: /tmp/cloudh_outflows_channel_maps/SiO_8-7_channel_maps_upsampled.fits


In [50]:
def make_rotated_display_wcs(native_wcs, coverage):
    boundary = coverage & ~binary_erosion(coverage)
    boundary_y, boundary_x = np.nonzero(boundary)
    if boundary_x.size == 0:
        raise ValueError('The channel maps have no finite footprint.')

    center_x = 0.5 * (boundary_x.min() + boundary_x.max())
    center_y = 0.5 * (boundary_y.min() + boundary_y.max())
    center = native_wcs.pixel_to_world(center_x, center_y)
    pixel_scale_deg = float(np.mean(proj_plane_pixel_scales(native_wcs)))

    rotation = np.deg2rad(DISPLAY_PA_DEG + 90)
    display_wcs = WCS(naxis=2)
    display_wcs.wcs.ctype = ['RA---TAN', 'DEC--TAN']
    display_wcs.wcs.cunit = ['deg', 'deg']
    display_wcs.wcs.crval = [center.ra.deg, center.dec.deg]
    display_wcs.wcs.crpix = [1.0, 1.0]
    display_wcs.wcs.cdelt = [pixel_scale_deg, pixel_scale_deg]
    display_wcs.wcs.pc = np.array([[-1, 0], [0, 1]]) @ np.array(
        [
            [np.cos(rotation), -np.sin(rotation)],
            [np.sin(rotation), np.cos(rotation)],
        ]
    )
    display_wcs.wcs.set()

    boundary_sky = native_wcs.pixel_to_world(boundary_x, boundary_y)
    display_x, display_y = display_wcs.world_to_pixel(boundary_sky)
    span_x = display_x.max() - display_x.min()
    span_y = display_y.max() - display_y.min()
    padding = max(4, int(np.ceil(0.02 * max(span_x, span_y))))
    shift_x = padding - display_x.min()
    shift_y = padding - display_y.min()
    display_wcs.wcs.crpix += [shift_x, shift_y]
    width = int(np.ceil(span_x + 2 * padding + 1))
    height = int(np.ceil(span_y + 2 * padding + 1))
    display_wcs.wcs.set()
    return display_wcs, (height, width), pixel_scale_deg * 3600


def sample_finite_map(data, input_x, input_y):
    valid = np.isfinite(data)
    sampled_values = map_coordinates(
        np.where(valid, data, 0.0), [input_y, input_x], order=1, mode='constant', cval=0.0
    )
    sampled_weights = map_coordinates(
        valid.astype(np.float32), [input_y, input_x], order=1, mode='constant', cval=0.0
    )
    sampled = np.full(input_x.shape, np.nan, dtype=np.float32)
    good = sampled_weights > 0.5
    sampled[good] = sampled_values[good] / sampled_weights[good]
    return sampled


def apply_stretch(data, scale, stretch):
    normalized = np.clip(np.nan_to_num(data, nan=0.0) / scale, 0.0, 1.0)
    if stretch == 'linear':
        return normalized
    if stretch == 'sqrt':
        return np.sqrt(normalized)
    return np.arcsinh(normalized / ASINH_SOFTENING) / np.arcsinh(1.0 / ASINH_SOFTENING)


def read_jwst_rgb(jpeg_path, wcs_path):
    rgb = np.flipud(imread(jpeg_path)[..., :3]).copy()
    with fits.open(wcs_path, memmap=True) as hdul:
        image_shape = hdul[1].data.shape
        image_wcs = WCS(hdul[1].header).celestial
    if rgb.shape[:2] != image_shape:
        raise ValueError(f'{jpeg_path.name} shape {rgb.shape[:2]} does not match {image_shape}.')
    return rgb, image_wcs


def sample_rgb(rgb, input_wcs, ra, dec):
    x, y = input_wcs.world_to_pixel_values(ra, dec)
    sampled = np.empty(ra.shape + (3,), dtype=rgb.dtype)
    for channel in range(3):
        sampled[..., channel] = map_coordinates(
            rgb[..., channel], [y, x], order=1, mode='constant', cval=0
        )
    return sampled


cache_hdul = fits.open(cache_path, memmap=True)
channel_maps = cache_hdul[0].data
cache_header = cache_hdul[0].header
channel_table = cache_hdul['CHANNELS'].data
if 'HASDATA' in channel_table.names:
    fine_map_has_data = np.asarray(channel_table['HASDATA'], dtype=bool)
else:
    fine_map_has_data = np.any(np.isfinite(channel_maps), axis=(1, 2))
native_wcs = WCS(cache_header).celestial
native_coverage = np.any(np.isfinite(channel_maps), axis=0)
display_wcs, display_shape, display_pixel_scale = make_rotated_display_wcs(
    native_wcs, native_coverage
)
display_y, display_x = np.mgrid[0:display_shape[0], 0:display_shape[1]]
display_ra, display_dec = display_wcs.pixel_to_world_values(display_x, display_y)
input_x, input_y = native_wcs.world_to_pixel_values(display_ra, display_dec)
display_coverage = map_coordinates(
    native_coverage.astype(np.float32), [input_y, input_x], order=0, mode='constant', cval=0.0
) > 0.5

nircam_rgb, nircam_wcs = read_jwst_rgb(NIRCAM_JPEG, NIRCAM_WCS_FITS)
miri_rgb, miri_wcs = read_jwst_rgb(MIRI_JPEG, MIRI_WCS_FITS)
miri_white = np.all(miri_rgb >= MIRI_WHITE_THRESHOLD, axis=2)
miri_edge = np.zeros(miri_white.shape, dtype=bool)
miri_edge[[0, -1], :] = True
miri_edge[:, [0, -1]] = True
miri_rgb[binary_propagation(miri_edge & miri_white, mask=miri_white)] = 0
display_nircam = sample_rgb(nircam_rgb, nircam_wcs, display_ra, display_dec)
display_miri = sample_rgb(miri_rgb, miri_wcs, display_ra, display_dec)
display_maps = np.empty((N_FINE_MAPS, *display_shape), dtype=np.float32)
for index in range(N_FINE_MAPS):
    display_maps[index] = sample_finite_map(channel_maps[index], input_x, input_y)
    print(f'Resampled fine map {index + 1:02d}/{N_FINE_MAPS}')

maximum_samples = 2_000_000
stride = max(1, int(np.ceil(np.sqrt(display_maps.size / maximum_samples))))
scale_samples = display_maps[:, ::stride, ::stride]
positive_samples = scale_samples[np.isfinite(scale_samples) & (scale_samples > 0)]
if positive_samples.size == 0:
    display_scale = 1.0
    print('No positive integrated emission is available; using a unit display scale.')
else:
    display_scale = np.percentile(positive_samples, UPPER_PERCENTILE)
print(f'Rotated display shape: {display_shape}; pixel scale={display_pixel_scale:.4f} arcsec/pixel')
print(f'{STRETCH} stretch: {UPPER_PERCENTILE:g}th-percentile scale={display_scale:.4g} K km/s')

Set OBSGEO-B to   -23.022886 from OBSGEO-[XYZ].
Set OBSGEO-H to     5053.796 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2026-05-12T02:20:38.473' from MJD-AVG.
Set DATE-END to '2026-05-12T02:31:06.605' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -29.337902 from OBSGEO-[XYZ].
Set OBSGEO-H to 1674445997.841 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2026-04-20T19:13:48.581' from MJD-AVG.
Set DATE-END to '2026-04-20T20:21:51.119' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -19.247298 from OBSGEO-[XYZ].
Set OBSGEO-H to 1505726645.939 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


Resampled fine map 01/56
Resampled fine map 02/56
Resampled fine map 03/56
Resampled fine map 04/56
Resampled fine map 05/56
Resampled fine map 06/56
Resampled fine map 07/56
Resampled fine map 08/56
Resampled fine map 09/56
Resampled fine map 10/56
Resampled fine map 11/56
Resampled fine map 12/56
Resampled fine map 13/56
Resampled fine map 14/56
Resampled fine map 15/56
Resampled fine map 16/56
Resampled fine map 17/56
Resampled fine map 18/56
Resampled fine map 19/56
Resampled fine map 20/56
Resampled fine map 21/56
Resampled fine map 22/56
Resampled fine map 23/56
Resampled fine map 24/56
Resampled fine map 25/56
Resampled fine map 26/56
Resampled fine map 27/56
Resampled fine map 28/56
Resampled fine map 29/56
Resampled fine map 30/56
Resampled fine map 31/56
Resampled fine map 32/56
Resampled fine map 33/56
Resampled fine map 34/56
Resampled fine map 35/56
Resampled fine map 36/56
Resampled fine map 37/56
Resampled fine map 38/56
Resampled fine map 39/56
Resampled fine map 40/56


In [ ]:
def velocity_token(value):
    prefix = 'm' if value < 0 else 'p'
    return f'{prefix}{abs(value):g}'.replace('.', 'p')


stretched_maps = np.array(
    [apply_stretch(channel_map, display_scale, STRETCH) for channel_map in display_maps],
    dtype=np.float32,
)
colour_hues = np.linspace(2 / 3, 0, N_PANEL_COLOURS)
colour_palette = hsv_to_rgb(
    np.column_stack((colour_hues, np.ones(N_PANEL_COLOURS), np.ones(N_PANEL_COLOURS)))
)
palette_normalization = colour_palette.sum(axis=0)


def make_panel_rgb(panel_index):
    central_start = panel_index * UPSAMPLE_FACTOR
    central_has_data = fine_map_has_data[central_start:central_start + UPSAMPLE_FACTOR]
    if not np.any(central_has_data):
        return np.zeros((*display_shape, 3), dtype=np.float32), 'none'

    data_status = 'partial' if not np.all(central_has_data) else 'full'
    first_fine_index = (panel_index - 1) * UPSAMPLE_FACTOR
    rgb = np.zeros((*display_shape, 3), dtype=np.float32)
    for colour_index, colour in enumerate(colour_palette):
        fine_index = first_fine_index + colour_index
        if 0 <= fine_index < N_FINE_MAPS and fine_map_has_data[fine_index]:
            rgb += stretched_maps[fine_index, ..., np.newaxis] * colour
    return np.clip(rgb / palette_normalization, 0.0, 1.0), data_status


bbox_style = dict(
    boxstyle='round,pad=0.28', facecolor='black', edgecolor='white',
    linewidth=0.8, alpha=0.72,
)


def style_sky_axis(ax):
    ax.set_facecolor('black')
    ax.coords.grid(color='white', alpha=0.35, linestyle=':')
    ax.coords[0].set_ticklabel_visible(False)
    ax.coords[1].set_ticklabel_visible(False)
    ax.coords[0].set_axislabel('')
    ax.coords[1].set_axislabel('')


fig = plt.figure(figsize=(11, 13.2), facecolor='white')
grid = fig.add_gridspec(6, 5)

# Two JWST overview panels occupy the upper-left grid slots.
for column, (image, instrument) in enumerate(
    ((display_nircam, 'NIRCam'), (display_miri, 'MIRI'))
):
    ax = fig.add_subplot(grid[0, column], projection=display_wcs)
    style_sky_axis(ax)
    ax.imshow(image, origin='lower', interpolation='nearest')
    ax.contour(
        display_coverage.astype(float), levels=[0.5], colors='white',
        linestyles='--', linewidths=1.2,
    )
    ax.text(
        0.03, 0.04, instrument, transform=ax.transAxes, ha='left', va='bottom',
        fontsize=9, fontweight='bold', color='white', bbox=bbox_style,
    )

# The 28 velocity panels fill every remaining grid slot.
for index in range(N_PANELS):
    rgb, data_status = make_panel_rgb(index)
    grid_slot = index + 2
    row = grid_slot // 5
    column = grid_slot % 5
    ax = fig.add_subplot(grid[row, column], projection=display_wcs)
    style_sky_axis(ax)
    ax.imshow(rgb, origin='lower', interpolation='nearest')
    ax.contour(
        display_coverage.astype(float), levels=[0.5], colors='white',
        linestyles='--', linewidths=1.0,
    )

    beam = Ellipse(
        (0.90 * display_shape[1], 0.08 * display_shape[0]),
        width=cache_header['BMAJ'] * 3600 / display_pixel_scale,
        height=cache_header['BMIN'] * 3600 / display_pixel_scale,
        angle=cache_header['BPA'] - DISPLAY_PA_DEG,
        facecolor='none', edgecolor='white', linewidth=1.2, alpha=0.9,
    )
    ax.add_patch(beam)

    lower = display_velocity_edges[index]
    upper = display_velocity_edges[index + 1]
    velocity_text = f'{lower:+.1f} to {upper:+.1f} km s$^{{-1}}$'
    annotation = f'{config["label"]}\n{velocity_text}' if index == 0 else velocity_text
    ax.text(
        0.03, 0.96, annotation, transform=ax.transAxes, ha='left', va='top',
        fontsize=8.5, fontweight='bold', color='white', bbox=bbox_style,
    )
    if data_status != 'full':
        status_text = 'No spectral data' if data_status == 'none' else 'Partial spectral coverage'
        ax.text(
            0.5, 0.5, status_text, transform=ax.transAxes, ha='center', va='center',
            fontsize=9, fontweight='bold', color='white', bbox=bbox_style,
        )

fig.subplots_adjust(left=0.01, right=0.99, bottom=0.01, top=0.99, wspace=0.02, hspace=0.02)
output_stem = (
    f'alma_channel_maps_rotated_{SELECTED_LINE}_'
    f'{velocity_token(VELOCITY_MIN)}_to_{velocity_token(VELOCITY_MAX)}_{N_PANELS}bins_'
    f'upsample{UPSAMPLE_FACTOR}_{N_PANEL_COLOURS}colour_jwst_overviews_whitebg'
)
for extension in ('pdf', 'png'):
    output_path = PLOT_DIR / f'{output_stem}.{extension}'
    fig.savefig(output_path, dpi=100, bbox_inches='tight', facecolor='white')
    print(f'Saved {output_path}')
plt.show()